# Story FLUX Runner on Colab GPU

This notebook copies the current repo into Colab from a zip, downloads FLUX.1-schnell weights from Hugging Face, and runs local GPU inference into the existing `pipeline/` folder.

Before opening this notebook in Colab, run `./make-colab-zip.ps1` on your Windows machine. Upload the generated `story-flux-runner-colab.zip` when prompted below.

In [ ]:
# Check that Colab has a GPU attached: Runtime > Change runtime type > T4/A100/L4 GPU
!nvidia-smi

In [ ]:
from google.colab import files
from pathlib import Path
import os, shutil, zipfile

repo_dir = Path('/content/story-flux-runner')
if repo_dir.exists():
    shutil.rmtree(repo_dir)

print('Upload story-flux-runner-colab.zip')
uploaded = files.upload()
zip_name = next(iter(uploaded))

with zipfile.ZipFile(zip_name) as zf:
    zf.extractall(repo_dir)

os.chdir(repo_dir)
print('Repo copied to:', repo_dir)
!find . -maxdepth 2 -type f | sort | head -80

In [ ]:
# Keep the token out of the notebook and out of the uploaded zip.
from getpass import getpass
import os

if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass('Paste your Hugging Face token: ')

assert os.environ['HF_TOKEN'].startswith('hf_'), 'HF_TOKEN should start with hf_'

In [ ]:
# Install local inference dependencies. Restart runtime only if Colab asks you to.
!pip -q install -U diffusers transformers accelerate safetensors huggingface_hub pillow sentencepiece protobuf

In [ ]:
import torch
from huggingface_hub import login, snapshot_download
from diffusers import FluxPipeline

login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)

model_id = 'black-forest-labs/FLUX.1-schnell'
weights_dir = '/content/hf-weights/FLUX.1-schnell'

snapshot_download(
    repo_id=model_id,
    local_dir=weights_dir,
    token=os.environ['HF_TOKEN'],
    ignore_patterns=['*.onnx', '*.msgpack', '*.bin'],
)

dtype = torch.float16
pipe = FluxPipeline.from_pretrained(weights_dir, torch_dtype=dtype)
pipe.enable_model_cpu_offload()
pipe.vae.enable_slicing()
pipe.vae.enable_tiling()

print('Loaded', model_id, 'on', torch.cuda.get_device_name(0))

In [ ]:
import gc, json, random
from pathlib import Path
from PIL import Image

story_path = Path('story.json')
story = json.loads(story_path.read_text())

IMAGE_WIDTH = 832
IMAGE_HEIGHT = 480
STEPS = 4
GUIDANCE_SCALE = 0.0
CANDIDATES_PER_SCENE = 1

# Empty list means all json-approved scenes. For a quick smoke test, use ['s06'].
SCENES_TO_RUN = ['s06']
RUN_CHARACTER_REFS = False
FORCE_REGEN = True

def as_posix(path):
    return str(path).replace('\\', '/')

def save_story():
    story_path.write_text(json.dumps(story, indent=2))

def run_flux(prompt, seed):
    generator = torch.Generator(device='cpu').manual_seed(int(seed))
    with torch.inference_mode():
        image = pipe(
            prompt=prompt,
            width=IMAGE_WIDTH,
            height=IMAGE_HEIGHT,
            num_inference_steps=STEPS,
            guidance_scale=GUIDANCE_SCALE,
            generator=generator,
            max_sequence_length=256,
        ).images[0]
    gc.collect()
    torch.cuda.empty_cache()
    return image

def char_prompt(char):
    return ', '.join([
        story['meta'].get('art_style', ''),
        char.get('flux_description', ''),
        'single character reference image, full body, centered, plain light background',
        'children storybook illustration, clean silhouette, no text, no watermark',
    ])

def scene_prompt(scene):
    bits = [
        story['meta'].get('art_style', ''),
        scene.get('flux_prompt', ''),
        scene.get('camera_framing', ''),
        f"mood: {scene.get('mood', '')}",
        '16:9 composition, no text, no watermark, no logo',
    ]
    return ', '.join([b for b in bits if b])

Path('pipeline/characters').mkdir(parents=True, exist_ok=True)
Path('pipeline/scenes').mkdir(parents=True, exist_ok=True)

In [ ]:
if RUN_CHARACTER_REFS:
    for char in story.get('characters', []):
        ref_path = Path('pipeline/characters') / f"{char['id']}_reference.png"
        if ref_path.exists() and not FORCE_REGEN:
            print('exists:', ref_path)
            continue

        seed = int(char.get('_ref_seed') or random.randint(0, 2_147_483_647))
        print('generating character ref:', char.get('name'), 'seed', seed)
        image = run_flux(char_prompt(char), seed)
        image.save(ref_path)
        char['reference_image_path'] = as_posix(ref_path)
        char['_ref_seed'] = seed
        save_story()
        display(image)

print('Character ref step complete')

In [ ]:
char_map = {c['id']: c for c in story.get('characters', [])}

scenes = [s for s in story.get('scenes', []) if s.get('review', {}).get('json_approved')]
if SCENES_TO_RUN:
    scenes = [s for s in scenes if s.get('scene_id') in SCENES_TO_RUN]

print('Scenes to run:', [s['scene_id'] for s in scenes])

for scene in scenes:
    scene_id = scene['scene_id']
    scene_dir = Path('pipeline/scenes') / scene_id
    scene_dir.mkdir(parents=True, exist_ok=True)
    scene.setdefault('outputs', {})
    candidates = scene['outputs'].setdefault('image_candidates', [])

    primary_char = None
    if scene.get('characters_present'):
        primary_char = char_map.get(scene['characters_present'][0])

    base_seed = scene.get('flux_seed')
    if base_seed is None:
        ref_seed = primary_char.get('_ref_seed') if primary_char else None
        base_seed = ((int(ref_seed) + int(scene.get('sequence', 1)) * 1000) % 2_147_483_647) if ref_seed else random.randint(0, 2_147_483_647)
        scene['flux_seed'] = base_seed

    print('\nscene:', scene_id, 'base seed:', base_seed)
    prompt = scene_prompt(scene)

    for i in range(CANDIDATES_PER_SCENE):
        seed = int(base_seed) + i
        img_path = scene_dir / f"c{i + 1}_seed{seed}.png"
        if img_path.exists() and not FORCE_REGEN:
            print('exists:', img_path)
            continue

        print('generating:', img_path)
        image = run_flux(prompt, seed)
        image.save(img_path)
        if not any(c.get('path') == as_posix(img_path) for c in candidates):
            candidates.append({'path': as_posix(img_path), 'seed': seed})
        display(image)

    save_story()

print('\nDone. Updated story.json and pipeline/scenes/.')

## Download Results

Run the next cell to zip the updated `story.json` and `pipeline/` outputs for download back to Windows.

In [ ]:
!zip -qr /content/story-flux-runner-results.zip story.json pipeline
files.download('/content/story-flux-runner-results.zip')